# Retail Flow Dataset

This notebook prepares the data for a secondary-market retail flow prediction model.

Each row in the position dataset represents the state of one client position at a specific point in time.

The goal is to combine position, product, market and client-behaviour information and create the target `sold_next_24h`, which indicates whether the client sells that product within the following 24 hours.

In [1]:
import pandas as pd
import numpy as np

In [2]:
positions = pd.read_csv("../data/synthetic_position_snapshots.csv")
transactions = pd.read_csv("../data/synthetic_transactions.csv")

In [3]:
print(positions.shape)
print(transactions.shape)
print(positions.columns)
print(transactions.columns)

(6717, 27)
(614, 6)
Index(['snapshot_id', 'timestamp', 'position_id', 'client', 'product_id',
       'product_type', 'underlyings', 'worst_underlying', 'notional',
       'issue_date', 'expiry_date', 'next_observation_date', 'days_to_expiry',
       'days_to_observation', 'holding_days', 'worst_performance',
       'barrier_pct', 'barrier_distance', 'autocall_pct', 'autocall_distance',
       'return_1d', 'return_5d', 'shock_magnitude', 'iv_change',
       'client_pnl_pct', 'historical_client_sell_rate',
       'recent_client_activity'],
      dtype='str')
Index(['transaction_id', 'timestamp', 'client', 'product_id', 'side',
       'notional'],
      dtype='str')


## Position snapshot columns

### Identifiers

- `snapshot_id`: unique ID for each position snapshot.
- `timestamp`: time at which the position is observed.
- `position_id`: identifies the client position across different snapshots.
- `client`: synthetic client identifier.
- `product_id`: structured product identifier.
- `product_type`: type of structured product.

### Product and position information

- `underlyings`: underlying or basket of underlyings linked to the product.
- `worst_underlying`: current worst-performing underlying in the basket.
- `notional`: current position size.
- `issue_date`: product issue date.
- `expiry_date`: product maturity date.
- `next_observation_date`: next autocall/observation date, where applicable.
- `days_to_expiry`: days remaining until maturity.
- `days_to_observation`: days remaining until the next observation date.
- `holding_days`: number of days the client has held the position.

### Product state

- `worst_performance`: performance of the worst underlying relative to its initial reference level.
- `barrier_pct`: barrier level expressed as a percentage of the initial reference level.
- `barrier_distance`: relative distance between the worst underlying and the barrier.
- `autocall_pct`: autocall level expressed as a percentage of the initial reference level.
- `autocall_distance`: relative distance between the worst underlying and the autocall level.

### Market state

- `return_1d`: 1-day return of the relevant worst underlying.
- `return_5d`: 5-day return of the relevant worst underlying.
- `shock_magnitude`: absolute value of the 1-day market move.
- `iv_change`: change in implied volatility.

### Client behaviour

- `client_pnl_pct`: synthetic estimate of the client's current P&L on the position.
- `historical_client_sell_rate`: historical tendency of the client to sell positions.
- `recent_client_activity`: recent number of client transactions.

In [4]:
positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 6717 entries, 0 to 6716
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   snapshot_id                  6717 non-null   str    
 1   timestamp                    6717 non-null   str    
 2   position_id                  6717 non-null   str    
 3   client                       6717 non-null   str    
 4   product_id                   6717 non-null   str    
 5   product_type                 6717 non-null   str    
 6   underlyings                  6717 non-null   str    
 7   worst_underlying             6717 non-null   str    
 8   notional                     6717 non-null   int64  
 9   issue_date                   6717 non-null   str    
 10  expiry_date                  6717 non-null   str    
 11  next_observation_date        2928 non-null   str    
 12  days_to_expiry               6717 non-null   int64  
 13  days_to_observation          

**Note:** Missing values are expected for features that do not apply to a given product type (e.g. autocall fields for non-autocall products).

In [5]:
positions.describe()

,notional,days_to_expiry,days_to_observation,holding_days,worst_performance,barrier_pct,barrier_distance,autocall_pct,autocall_distance,return_1d,return_5d,shock_magnitude,iv_change,client_pnl_pct,historical_client_sell_rate,recent_client_activity
count,6.717000e+03,6717.000000,2928.000000,6717.000000,6717.000000,5185.000000,5185.000000,2928.0,2928.000000,6717.000000,5707.000000,6717.000000,6717.000000,6717.000000,6717.000000,6717.000000
mean,8.519577e+05,296.432634,13.304303,101.151258,0.936317,0.680733,0.397740,1.0,-0.092412,-0.002617,-0.011257,0.018130,0.001665,-0.029047,0.267495,1.572428
std,6.578937e+05,45.930254,7.821109,46.273960,0.179093,0.053675,0.254783,0.0,0.181472,0.027384,0.059908,0.020688,0.023716,0.101298,0.120420,1.518891
min,1.000000e+05,114.000000,0.000000,10.000000,0.542058,0.600000,-0.104197,1.0,-0.457942,-0.180243,-0.257858,0.000001,-0.050000,-0.384386,0.052632,0.000000
25%,2.500000e+05,267.000000,7.000000,66.000000,0.833806,0.650000,0.233151,1.0,-0.191365,-0.011906,-0.042186,0.005064,-0.011158,-0.086616,0.166667,1.000000
50%,7.500000e+05,301.000000,13.000000,99.000000,0.898320,0.700000,0.358010,1.0,-0.123569,0.000366,-0.005636,0.011702,-0.000406,-0.043733,0.263158,1.000000
75%,1.500000e+06,329.000000,20.000000,129.000000,0.991315,0.750000,0.487211,1.0,-0.068022,0.011362,0.025139,0.023680,0.011764,0.006220,0.360000,2.000000
max,2.000000e+06,383.000000,27.000000,239.000000,1.572714,0.750000,1.246734,1.0,0.564925,0.101723,0.220539,0.180243,0.120000,0.300000,0.533333,9.000000


In [6]:
positions.isna().sum().sort_values(ascending=False)

autocall_pct                   3789
days_to_observation            3789
next_observation_date          3789
autocall_distance              3789
barrier_distance               1532
barrier_pct                    1532
return_5d                      1010
client                            0
snapshot_id                       0
timestamp                         0
position_id                       0
expiry_date                       0
issue_date                        0
notional                          0
worst_underlying                  0
underlyings                       0
product_type                      0
product_id                        0
days_to_expiry                    0
worst_performance                 0
holding_days                      0
return_1d                         0
shock_magnitude                   0
iv_change                         0
client_pnl_pct                    0
historical_client_sell_rate       0
recent_client_activity            0
dtype: int64

In [7]:
# Product distribution
positions["product_type"].value_counts()

product_type
Callable Multi BRC             1736
Barrier Reverse Convertible    1679
Autocall                       1192
Tracker                         778
Reverse Convertible             754
Bonus Certificate               578
Name: count, dtype: int64

In [8]:
# Universe size
print("Clients:", positions["client"].nunique())
print("Products:", positions["product_id"].nunique())
print("Positions:",positions["position_id"].nunique())
print("Snapshots:", len(positions))


Clients: 20
Products: 36
Positions: 211
Snapshots: 6717


In [9]:
# Time period
print(positions["timestamp"].min())
print(positions["timestamp"].max())

2026-01-05 09:00:00
2026-06-19 09:00:00


In [10]:
# Convert date columns
date_columns = ["timestamp", "issue_date", "expiry_date", "next_observation_date"]

for column in date_columns:
    positions[column] = pd.to_datetime(positions[column])

# For transactions df as well
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

In [11]:
# Transactions overview
print(transactions.describe())

                        timestamp      notional
count                         614  6.140000e+02
mean   2025-12-13 16:29:04.299674  8.519544e+05
min           2025-09-07 11:00:00  1.000000e+05
25%           2025-10-29 21:07:30  2.500000e+05
50%           2025-12-05 01:45:00  7.500000e+05
75%           2026-01-16 15:30:00  1.500000e+06
max           2026-06-16 15:30:00  2.000000e+06
std                           NaN  6.378206e+05


In [12]:
transactions["side"].value_counts()

side
BUY     336
SELL    278
Name: count, dtype: int64

## Target: sold in the next 24 hours

For each position snapshot, check whether the same client sells the same product within the following 24 hours.

Future transactions are used only to create the target, not as model features.

In [13]:
sell_transactions = transactions[transactions["side"]=="SELL"].copy()

sell_transactions = sell_transactions.rename(columns={"timestamp":"sell_timestamp"})

In [14]:
snapshot_sales = positions[["snapshot_id","timestamp", "client", "product_id"]].merge(
    sell_transactions[["sell_timestamp","client","product_id"]],
    on = ["client","product_id"],
    how = "left"
)

snapshot_sales["sold_next_24h"]= ( 
    (snapshot_sales["sell_timestamp"] > snapshot_sales["timestamp"])
    &
    (snapshot_sales["sell_timestamp"] <= snapshot_sales["timestamp"] + pd.Timedelta(hours=24) )
    )

snapshot_sales.head()

,snapshot_id,timestamp,client,product_id,sell_timestamp,sold_next_24h
0,S000001,2026-01-05 09:00:00,Client_01,P024,2026-02-24 15:30:00,False
1,S000002,2026-01-05 09:00:00,Client_01,P028,2026-04-02 15:30:00,False
2,S000003,2026-01-05 09:00:00,Client_01,P004,2026-03-24 15:30:00,False
3,S000004,2026-01-05 09:00:00,Client_01,P011,2026-02-16 15:30:00,False
4,S000005,2026-01-05 09:00:00,Client_01,P020,2026-01-30 15:30:00,False


In [15]:
# we want one row per snapshot

target = (snapshot_sales.groupby("snapshot_id")["sold_next_24h"].max().astype(int).reset_index())
len(target)

6717

In [16]:
positions = positions.merge(target, on = "snapshot_id", how="left")

#check
positions["sold_next_24h"].value_counts()

sold_next_24h
0    6507
1     210
Name: count, dtype: int64

In [17]:
positions["sold_next_24h"].value_counts(normalize=True)

sold_next_24h
0    0.968736
1    0.031264
Name: proportion, dtype: float64

In [18]:
positions.to_csv(
    "../data/synthetic_retail_flow_dataset.csv",
    index=False
)

In [19]:
positions[[
    "snapshot_id",
    "timestamp",
    "client",
    "product_id",
    "sold_next_24h"
]].head()

,snapshot_id,timestamp,client,product_id,sold_next_24h
0,S000001,2026-01-05 09:00:00,Client_01,P024,0
1,S000002,2026-01-05 09:00:00,Client_01,P028,0
2,S000003,2026-01-05 09:00:00,Client_01,P004,0
3,S000004,2026-01-05 09:00:00,Client_01,P011,0
4,S000005,2026-01-05 09:00:00,Client_01,P020,0
